# analysis.detection-statistics-virus


## Summary 

In this notebook, we focus on the numbers generated by the previous virus-taxonomic profiling using a BLAST-based pipeline. We read mostly content from the `metadata.site-library.csv` files and from `hits.virus.csv`. 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import taxoniq
from daforfer import DaforferDB
plt.rcParams['svg.fonttype'] = 'none'
from yaml import load, Loader
from daforfer import DaforferDB
conf = load(open("conf.yaml"), Loader)
db = DaforferDB(conf['database'])
db.toc()

## Bacteria

We carry out the analysis here by making questions and answering them.

### How many bacteria were detected?

To know that, first we need to load the data. 

In [ ]:
virus_hits = db.conn.sql('SELECT * FROM D_virusHits').df()
metadata = db.conn.sql('SELECT * FROM D_sites').df()

In [ ]:
virus_hits

In [ ]:
len(virus_hits.acronym.unique())

In [ ]:
len(virus_hits.scientific_name.unique())

Now, we value count the different taxids. 

In [ ]:
virus_hits = pd.merge(metadata, virus_hits, on='library', how='left').dropna(subset='scientific_name')
# virus_hits['taxid'] = virus_hits['taxid'].astype(int)
virus_hits

The data contains hits. We need to aggregate the hits to get the OTUs to which they map, which we do by a simple `value_counts`. 

In [ ]:
virus_hits.value_counts(['scientific_name']).reset_index()

In [ ]:
virus_hits_count = virus_hits.value_counts(['scientific_name']).reset_index()
virus_hits_count['rank'] = 1 + np.arange(len(virus_hits_count))
virus_hits_count['%'] = virus_hits_count['count'] / len(virus_hits)
virus_hits_count

In [ ]:
print("Total hits {:6d}".format(len(virus_hits))) # 1619
print(" |--> corresponding to {:6d} virus OTUs".format(len(virus_hits_count))) # 158


We have 1619 hits corresponding to 158 OTUs. Now, let's see how they distribute. 

In [ ]:
pd.DataFrame.from_records(
    [
        {"threshold": "=1", "count": len(virus_hits_count.query('count == 1'))}, # 53
        {"threshold": ">1", "count": len(virus_hits_count.query('count > 1'))}, # 105
        {"threshold": ">2", "count": len(virus_hits_count.query('count > 2'))}, # 77 
        {"threshold": ">5", "count": len(virus_hits_count.query('count > 5'))}, # 43
        {"threshold": ">10", "count": len(virus_hits_count.query('count > 10'))}, # 27
    ]
)



Most OTUs (> 100) have been detected more than once. A ranking map of these OTUs should help us see this pattern. In the folllowing plot, we are sorting by growing rank the OTUs depending on their number of hits. On the left, we would find the OTUs with higher number of hits, on the right those with the lower number of hits. 

In [ ]:
db.save_dataframe(
    virus_hits_count, table_name="viral_rank_plot",
    description="All virus OTUs ordered by their number of hits"
)

g = sns.relplot(
    data=virus_hits_count, y='count', x='rank', kind='line',
    height=2.0, aspect=2.5
)
# g.ax.axvline(10, linestyle='--', color='gray')
g.ax.set_yscale('log')


We can observe indeed a very heterogeneous distribution, with some hits exceeding the 100 detections. Let's see which could be these organisms.

In [ ]:
g = sns.catplot(
    data=virus_hits_count[:20], y='scientific_name', x='%', kind='bar', height=4.0, aspect=1.0
)
g.ax.set_xscale('log')

### How are the different PABs distributed?

We will generate a file that can be used as a table in the supplementary material, where bacteria are associated to their host ranges, PAB type, and number of hits. 

First, we will compute the host range, using the `host_taxon`, `taxid` and `scientific_name` as keys, and then counting by `scientific_name` and `taxid`. This should tell us how many different hosts can each entry find.  

In [ ]:
virus_host_range = virus_hits.value_counts(
    ['host_taxon', 'scientific_name']
    ).reset_index().value_counts(
        ['scientific_name']
    ).reset_index().rename(columns={'count': 'host_range'})
# assert(len(virus_host_range) == 127) # INTRODUCING THIS KIND OF CHECKS CAN HELP US CHECKING OUR OPERATIONS
virus_host_range

Now, we will count the number of hits per habitat. To do so, we simply do a value counts of the hits sharing `taxid,scientific_name,habitat`, and then we pivot the dataset to get it columns.

In [ ]:
virus_perhabitat = virus_hits.value_counts(["scientific_name", "habitat"]).reset_index() 
virus_perhabitat = virus_perhabitat.pivot(index=['scientific_name'], columns='habitat', values='count').fillna(0).reset_index()
virus_perhabitat


A slightly more complex query consists on getting the host-ranges per habitat. To do so, we need to first deduplicate the bacteria-hit dataset by `scientific_name,taxid,host_taxon,habitat`, and then repeat. 

In [ ]:
virus_hrperhabitat = virus_hits.drop_duplicates(
    ["scientific_name", "host_taxon", "habitat"]
).value_counts(
    ["scientific_name", "habitat"]
).reset_index() 
virus_hrperhabitat = virus_hrperhabitat.pivot(index=['scientific_name'], columns='habitat', values='count').fillna(0).reset_index()
virus_hrperhabitat


Now, we will compute site-ranges: the number of different sites that are reached by each virus.

In [ ]:
virus_sr = virus_hits.drop_duplicates(
    ["scientific_name", "site"]
).value_counts(
    ["scientific_name"]
).reset_index().rename(columns={'count': 'site_range'})
# bacteria_pab_sr = bacteria_pab_sr.pivot(index=['scientific_name', 'taxid', ], columns='site', values='count').fillna(0).reset_index()

virus_sr


Finally, we compute the number of habitats

In [ ]:
virus_habitatr = virus_hits.drop_duplicates(
    ["scientific_name", "habitat"]
).value_counts(
    ["scientific_name"]
).reset_index().rename(columns={'count': 'habitat_range'})
# bacteria_pab_sr = bacteria_pab_sr.pivot(index=['scientific_name', 'taxid', ], columns='site', values='count').fillna(0).reset_index()
virus_habitatr


Now, we can try to merge it all.

In [ ]:
virus_distribution = pd.merge(
    virus_hits.drop_duplicates(['scientific_name'])[['scientific_name', 'taxid']],
    virus_hits.value_counts(['scientific_name']).reset_index().rename(columns={'count': 'hits'})[['scientific_name', 'hits']],
    on=['scientific_name']
)

virus_distribution = pd.merge(
    virus_distribution,
    virus_habitatr,
    on=['scientific_name']
)

virus_distribution = pd.merge(
    virus_distribution,
    virus_host_range,
    on=['scientific_name']
)

virus_distribution = pd.merge(
    virus_distribution, virus_sr, on=['scientific_name']
)

virus_distribution = pd.merge(
    virus_distribution, virus_hrperhabitat.rename(columns={'Crop': 'Crop_HR', 'Edge': 'Edge_HR', 'Oak': 'Oak_HR', 'Wasteland': 'Wasteland_HR'}), 
    on=['scientific_name']
)

virus_distribution = pd.merge(
    virus_distribution, virus_perhabitat.rename(columns={'Crop': 'Crop_hits', 'Edge': 'Edge_hits', 'Oak': 'Oak_hits', 'Wasteland': 'Wasteland_hits'}), 
    on=['scientific_name']
)

virus_distribution

#### Testing

We have performed many operations at once. It is therefore convenient to check that the values that we are observing add up. Below, we will run a few tests.

In [ ]:
# TEST 1: No organism should have a host range larger than the host range that we had measured 
# at one of the habitats. 
for i in range(len(virus_distribution)):
    example = virus_distribution.loc[i]
    try:
        assert(max([example.Crop_HR,example.Edge_HR, example.Oak_HR,example.Wasteland_HR]) <= example.host_range)
    except AssertionError: 
        print("{0} did not pass the test 1".format(example['scientific_name']))

In [ ]:
for i in range(len(virus_distribution)):
    example = virus_distribution.loc[i]
    example_subset = virus_hits.query('scientific_name == "{0}"'.format(example.scientific_name))
    print(i)
    assert(len(example_subset) == int(example.Crop_hits + example.Edge_hits + example.Wasteland_hits + example.Oak_hits))
    assert(len(example_subset) == int(example.hits))
    assert(len(example_subset.drop_duplicates(['host_taxon'])) == example.host_range)
print("All tests passed! :-D")

#### Output

This table will be quite useful, so we will save it in a few different formats:

1. CSV to open it later.
2. Excel, to create a single multi-sheet document that eases the writing and the analysis by third parties.

In [ ]:
db.save_dataframe(
    virus_distribution, table_name="D_VirusOTUs",
    description="This table summarizes most of the information of previously detected virus OTUs, including host_range, site_range, habitat_range, etc."
)
# virus_distribution.sort_values(by='scientific_name').to_csv("output/table.virus-distribution.csv", sep=";")
# with pd.ExcelWriter('output/MIRIPVIR25.xlsx') as writer:  
#     pab_distribution.sort_values(by='scientific_name').to_excel(writer, sheet_name='PAB-distribution')

We could test these questions statistically, but in general if there is any effect, it seems weak. 

## Libraries

Again, simple Q & A

### What is the distribution of hits across our libraries?

To answer this question, first we need to value-count the number of total hits per library. We all save the habitat. 

In [ ]:
# library_hits_count = bacteria_hits.value_counts(['library', 'site', 'habitat']).reset_index()
# library_hits_count['rank'] = np.arange(len(library_hits_count))
# library_hits_count

The following table indicates some values of the distribution.

In [ ]:
# pd.DataFrame.from_records(
#     [
#         {"threshold": "=1", "count": len(library_hits_count.query('count == 1'))}, #88
#         {"threshold": ">1", "count": len(library_hits_count.query('count > 1'))}, #211
#         {"threshold": ">2", "count": len(library_hits_count.query('count > 2'))},
#         {"threshold": ">5", "count": len(library_hits_count.query('count > 5'))},
#         {"threshold": ">10", "count": len(library_hits_count.query('count > 10'))},
#     ]
# )



We have 299 positive libraries out of the 323 libraries, which means 24 negatives. Only 22 libraries have more than 10 OTUs detected.

We use the following rank plot to visualize the heterogeneity of host ranges.

In [ ]:
# res_ = library_hits_count.sort_values(by='count', ascending=False).query('count > 0')
# res_['rank'] = np.arange(1, len(res_) + 1)

# db.save_dataframe(
#     res_, table_name="site_rank_plot",
#     description="Ranking of sites depending on their detections"
# )

# g = sns.relplot(
#     data=res_,
#     x='rank', y='count', kind='line',
#     height=2.0, aspect=2.0
# )
# g.ax.set_yscale('log')
# g.set_xlabels("rank")
# g.set_ylabels("# hits per library")

In [ ]:
db.conn.close()